# 🎭 Servidor F5-TTS de BorGPT para Google Colab (1-Click Run)
Este notebook levanta un microservicio FastAPI gratuito con GPU T4 para clonar la voz de **Jorge Luis Borges** en tiempo real y alimentar la cabina teatral de **BorGPT**.

### 🚀 Instrucciones (¡Todo 100% automático!):
1. Asegúrate de tener seleccionada la GPU en **Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU**.
2. Ve a **Entorno de ejecución > Ejecutar todas** (Ctrl + F9).
3. El notebook descargará automáticamente la muestra de audio de Borges desde GitHub y levantará el túnel en `https://borges.alexruiz.ar`.

In [ ]:
# 1. Instalar dependencias necesarias
!pip install --quiet f5-tts fastapi uvicorn pydantic soundfile torch torchaudio

In [ ]:
# 2. Descarga automática del audio de referencia de Borges desde GitHub
import os
import urllib.request

REF_AUDIO = 'borges_reference.wav'
GITHUB_AUDIO_URL = 'https://raw.githubusercontent.com/alexruix/BorGPT/main/data/audio/borges_reference.wav'

if not os.path.exists(REF_AUDIO):
    print('⬇️ Descargando muestra de audio de Borges desde el repositorio de GitHub...')
    try:
        urllib.request.urlretrieve(GITHUB_AUDIO_URL, REF_AUDIO)
        print('✅ borges_reference.wav descargado automáticamente con éxito.')
    except Exception as e:
        print(f'⚠️ No se pudo descargar desde GitHub ({e}). Si estás en un repo privado, arrastra el archivo manualmente.')
else:
    print('✅ Archivo borges_reference.wav listo en el entorno.')

In [ ]:
# 3. Cargar modelo F5-TTS y levantar servidor permanente en https://borges.alexruiz.ar
import base64
import io
import os
import re
import subprocess
import time
import soundfile as sf
import torch
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from f5_tts.api import F5TTS

# =========================================================================
# 🔑 CONFIGURACIÓN PERMANENTE DE CLOUDFLARE PARA borges.alexruiz.ar
# =========================================================================
CLOUDFLARE_TUNNEL_TOKEN = "eyJhIjoiN2ZiYmJiMTRmYjQzMWI5NjhmODY3MmMxNGZkOTRhYzQiLCJ0IjoiNmM0MjA3M2MtMWIxMC00MjQ3LWE4ODYtZGRkNGUzM2U2NjU3IiwicyI6Ik9XRXpOMk13T0dRdFptUTNNQzAwTkRKbUxUaGpNV1F0Wm1Sa056SXlNemsyTlRKayJ9"
FIXED_DOMAIN = "https://borges.alexruiz.ar"
# =========================================================================

print("⏳ Cargando modelo F5-TTS en memoria de GPU...")
tts_engine = F5TTS()
print("✅ Modelo F5-TTS cargado exitosamente en GPU.")

app = FastAPI(title="BorGPT F5-TTS Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class TTSRequest(BaseModel):
    text: str
    speed: Optional[float] = 0.95

REF_AUDIO = "borges_reference.wav"
REF_TEXT = "que soy un lector hedónico. No he leído jamás un libro porque ese libro fuera antiguo, no he leído jamás un libro porque ese libro fuera contemporáneo. He leído..."

@app.get("/health")
def health():
    return {"status": "online", "model": "F5-TTS Borges Clone"}

@app.post("/tts")
async def generate_speech(req: TTSRequest):
    if not req.text.strip():
        raise HTTPException(status_code=400, detail="El texto no puede estar vacío")
    try:
        speed_val = req.speed if req.speed is not None else 0.95
        wav, sr, _ = tts_engine.infer(
            ref_file=REF_AUDIO,
            ref_text=REF_TEXT,
            gen_text=req.text,
            speed=speed_val,
        )
        buffer = io.BytesIO()
        sf.write(buffer, wav, sr, format='WAV')
        buffer.seek(0)
        b64_audio = base64.b64encode(buffer.read()).decode('utf-8')
        return {"audio_base64": b64_audio, "speed": speed_val}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Descargar binario de Cloudflare Linux para Colab
if not os.path.exists('cloudflared'):
    !wget --quiet https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

import threading
import uvicorn

def run_server():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# Iniciar túnel de Cloudflare permanente
token_clean = CLOUDFLARE_TUNNEL_TOKEN.strip()
if token_clean:
    print("🚀 Conectando túnel persistente con tu subdominio borges.alexruiz.ar...")
    tunnel_proc = subprocess.Popen(['./cloudflared', 'tunnel', 'run', '--token', token_clean], stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
    time.sleep(4)
    public_url = FIXED_DOMAIN
else:
    print("ℹ️ Modo temporal: generando túnel gratuito en *.trycloudflare.com...")
    tunnel_proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
    public_url = None
    start_time = time.time()
    while time.time() - start_time < 30:
        line = tunnel_proc.stderr.readline()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
        time.sleep(0.1)

if public_url:
    print('\n' + '='*65)
    print('🎭 ¡SERVIDOR TTS DE BORGES ONLINE CON SUBDOMINIO FIJO!')
    print(f'👉 DOMINIO PERMANENTE:  {public_url}')
    print('='*65 + '\n')
    print('✅ Conexión permanente con GPU activa. Tu .env ya está configurado.')
else:
    print('No se pudo extraer la URL. Revisa los logs de Cloudflare.')